In [134]:
# Loading + chunking
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector store
from langchain_chroma import Chroma

# LLM (local, via Ollama)
from langchain_ollama import OllamaLLM


from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate

from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever


In [14]:
path = "/home/therealgone/Projects/Brain-Rag/memory.txt"
loader = TextLoader(file_path=path)

docs = loader.load()
for doc in docs:
    print(doc.model_dump())

{'id': None, 'metadata': {'source': '/home/therealgone/Projects/Brain-Rag/memory.txt'}, 'page_content': "My name is Alex and I'm a backend developer working mostly with Python and Go. I've been coding professionally for about 4 years now, starting out at a small fintech startup before moving to a larger e-commerce company last year.\n\nI want to watch the movie Interstellar this weekend. I've heard really good things about the visuals and the sound design. A few of my friends said the ending confused them, so I should probably watch it when I can focus without distractions.\n\nFor my workout routine, I go to the gym three times a week, usually Monday, Wednesday, and Friday. I focus on compound lifts, squats, deadlifts, and bench press. My current goal is to hit a 100kg bench press by the end of the year.\n\nI'm currently learning how to cook Italian food properly. Last week I made a carbonara from scratch and it turned out too watery, I think I added the egg mixture while the pan was t

In [16]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=100,
    chunk_overlap=0,
)

chunks= splitter.split_documents(docs)

for i, chunk in enumerate(chunks):

    print(f"--- Chunk {i} ---")
    print(chunk.page_content)
    print()


Created a chunk of size 229, which is longer than the specified 100
Created a chunk of size 242, which is longer than the specified 100
Created a chunk of size 231, which is longer than the specified 100
Created a chunk of size 280, which is longer than the specified 100
Created a chunk of size 243, which is longer than the specified 100
Created a chunk of size 259, which is longer than the specified 100


--- Chunk 0 ---
My name is Alex and I'm a backend developer working mostly with Python and Go. I've been coding professionally for about 4 years now, starting out at a small fintech startup before moving to a larger e-commerce company last year.

--- Chunk 1 ---
I want to watch the movie Interstellar this weekend. I've heard really good things about the visuals and the sound design. A few of my friends said the ending confused them, so I should probably watch it when I can focus without distractions.

--- Chunk 2 ---
For my workout routine, I go to the gym three times a week, usually Monday, Wednesday, and Friday. I focus on compound lifts, squats, deadlifts, and bench press. My current goal is to hit a 100kg bench press by the end of the year.

--- Chunk 3 ---
I'm currently learning how to cook Italian food properly. Last week I made a carbonara from scratch and it turned out too watery, I think I added the egg mixture while the pan was too hot. I need to remember to take the pan off 

In [17]:
import json , os 

Categories_path= "/home/therealgone/Projects/Brain-Rag/categories.json"

try:
    with open(Categories_path, "r") as f:
        categories = json.load(f)
    if not isinstance(categories, list):
        raise ValueError
except (FileNotFoundError, json.JSONDecodeError, ValueError):
    categories = []
    with open(Categories_path, "w") as f:
        json.dump(categories, f)

print(categories)

['Fitness', 'PersonalLife', 'HardSciFiReading', 'CookingTips', 'TravelPlanning', 'Work', 'Movies', 'Books', 'Travel', 'Health']


In [18]:
from langchain_core.prompts import PromptTemplate

categorize_prompt = PromptTemplate.from_template("""
You are categorizing a personal memory note into a broad topic category.

Existing categories: {categories}

Memory text:
\"\"\"
{chunk_text}
\"\"\"

Instructions:
- Categories should be broad and general, not specific or narrow. Think top-level life areas, not sub-topics.
  Good examples: Health, Fitness, Work, Learning, Projects, Travel, Movies, Books, Food, Relationships, Finance.
  Bad examples (too specific): "Italian Cooking Mistakes", "Interstellar Review", "Bench Press Progress".
- Always check the existing categories first. If the memory reasonably fits one of them, even loosely, use that exact existing category name rather than creating a similar new one.
- Only invent a new category if the memory genuinely doesn't fit any existing one. New categories should be 1-2 words, Title Case, and just as broad as the examples above.
- Respond with ONLY the category name. No explanation, no punctuation, no extra text.
""")

In [28]:
def save_category_if_new(category, categories):
    if category not in categories:
        categories.append(category)
        with open(Categories_path, "w") as f:
            json.dump(categories, f, indent=2)
        print(f"Added new category: {category}")
    else:
        print(f"Matched existing category: {category}")
    return categories

categories = save_category_if_new(category, categories)


Matched existing category: Health


In [20]:
llm = OllamaLLM(model="phi4-mini")   # check `ollama list` for the exact tag you pulled

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma(
    collection_name="memories",
    embedding_function=embeddings,
    persist_directory="/home/therealgone/Projects/Brain-Rag/chroma_db",
)

for i, chunk in enumerate(chunks):

    prompt_text = categorize_prompt.format(
        categories=categories if categories else "None yet",
        chunk_text=chunk.page_content,
    )

    response = llm.invoke(prompt_text)
    category = response.strip()
    print(repr(category))
    categories = save_category_if_new(category, categories)

    chunk.metadata["category"] = category
    chunk_id = f"{os.path.basename(path)}_{i}"

    vectorstore.add_documents([chunk], ids=[chunk_id])
    print(f"Stored chunk {i} in Chroma with id={chunk_id!r}, category={category!r}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

'Work'
Matched existing category: Work
Stored chunk 0 in Chroma with id='memory.txt_0', category='Work'
'Movies'
Matched existing category: Movies
Stored chunk 1 in Chroma with id='memory.txt_1', category='Movies'
'Health'
Matched existing category: Health
Stored chunk 2 in Chroma with id='memory.txt_2', category='Health'
'CookingTips'
Matched existing category: CookingTips
Stored chunk 3 in Chroma with id='memory.txt_3', category='CookingTips'
'Books'
Matched existing category: Books
Stored chunk 4 in Chroma with id='memory.txt_4', category='Books'
'Travel'
Matched existing category: Travel
Stored chunk 5 in Chroma with id='memory.txt_5', category='Travel'
'Health'
Matched existing category: Health
Stored chunk 6 in Chroma with id='memory.txt_6', category='Health'


In [118]:
rewrite_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Rewrite the user's question into a clear, grammatically correct, and specific question for retrieval.

Rules:
- Preserve the exact meaning and intent of the original question.
- Do NOT add facts, topics, entities, context, assumptions, or interpretations that are not explicitly present in the question.
- Do NOT guess what the user means.
- Do NOT answer the question.
- Do NOT expand vague questions with invented context.
- If the original question is already clear, return it with only minor grammatical improvements.
- If the question is vague, improve its wording while keeping the same level of ambiguity.
- Preserve important words, names, entities, and terminology from the original question.
- Only use information contained in the original question.
- The rewritten question may be identical to the original if no meaningful clarification is possible.


Question: {question}

Rewritten question:"""
)

select_categories = PromptTemplate(
    input_variables=["categories", "question"],
    template="""Given the question below, do two things:
1. Extract the single most important keyword from the question.
2. Choose the best matching category from this list: {categories}
   If none fit well, invent a short new category name.

Question: {question}

Respond ONLY with valid JSON in this exact format, no other text:
{{"keyword": "...", "category": "..."}}"""
)

In [86]:
def keyword_search(keyword,vectorstore,k=4):
    all_data = vectorstore.get()  
    
    documents = all_data["documents"]
    metadatas = all_data["metadatas"]
    
    matches = []
    for doc, meta in zip(documents, metadatas):
        if keyword.lower() in doc.lower():   
            matches.append(doc)
    
    return matches   



               
               


In [110]:
answer_prompt = PromptTemplate.from_template(
    """You are answering a question using the user's personal memory notes below.

Memory excerpts:
{context}

Question: {question}

Instructions:
- Answer using only the information in the memory excerpts above.
- If the excerpts don't contain enough information to answer, say "I don't have a memory about that."
- Be concise and direct. """)

In [145]:
import json
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

def retrieve(question, k=3):
    # Step 1: rewrite the vague question
    rewrite_text = rewrite_prompt.format(question=question)

    rewritten_question = llm.invoke(rewrite_text).strip()

    print("new q", rewritten_question)
    # Step 2: get keyword + category together, from the rewritten question
    combined_text = select_categories.format(
        categories=categories if categories else "None yet",
        question=rewritten_question,
    )
    raw_response = llm.invoke(combined_text).strip()

    try:
        parsed = json.loads(raw_response)
        predicted_keyword = parsed.get("keyword", "")
        predicted_category = parsed.get("category", "")
    except json.JSONDecodeError:
        print("Failed to parse JSON, falling back to unfiltered search")
        predicted_keyword = ""
        predicted_category = ""

    print("BM")
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

    # bm25 retriever, built from the same chunked documents
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 3

    # combine them, weights control how much each contributes
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[0.6, 0.4],
        c=20  # tune this, e.g. 0.4/0.6 if one is more reliable
    )

    results_b = ensemble_retriever.invoke(rewritten_question)
    print("BM")

    results = []

    # try category-filtered search first
    if predicted_category in categories:
        results = vectorstore.similarity_search_with_score(
            rewritten_question, k=k, filter={"category": predicted_category}
        )

    # fall back to keyword search if category search gave nothing
    if not results and predicted_keyword:
        print("keyword")
        results = keyword_search(predicted_keyword, vectorstore, k=k)

    # final fallback, plain unfiltered similarity search
    if not results:
        print("Falling back to unfiltered search")
        results = vectorstore.similarity_search_with_score(rewritten_question, k=k)

# combine both lists into one list of Document objects first
    combined_docs = results_b + [doc for doc, score in results]

# now build context string from page_content
    context = "\n\n".join([doc.page_content for doc in combined_docs])

    response = answer_prompt.format(context=context, question=rewritten_question)
    answer = llm.invoke(response)
    print(answer)



In [ ]:
def answer_question(question, k=3):
    docs, docs_b = retrieve(question, k)
    context = "\n\n".join(d.page_content for d, score in docs)
    prompt_text = answer_prompt.format(context=context, question=question)
    response = llm.invoke(prompt_text)

    return response.strip(), docs

[Document(id='memory.txt_0', metadata={'category': 'Work', 'source': '/home/therealgone/Projects/Brain-Rag/memory.txt'}, page_content="My name is Alex and I'm a backend developer working mostly with Python and Go. I've been coding professionally for about 4 years now, starting out at a small fintech startup before moving to a larger e-commerce company last year."),
 Document(metadata={'source': '/home/therealgone/Projects/Brain-Rag/memory.txt', 'category': 'Health'}, page_content='I have a dentist appointment coming up on the 15th of next month for a routine cleaning. I keep forgetting to floss regularly, I should actually start doing that daily instead of only before dentist visits.'),
 Document(metadata={'source': '/home/therealgone/Projects/Brain-Rag/memory.txt', 'category': 'Travel'}, page_content="I'm planning a trip to Japan next spring, tentatively around late March to catch the cherry blossoms. I want to visit Tokyo, Kyoto, and maybe Osaka for the food scene. I still need to lo

In [147]:
q="i booked an appointment what was that"
retrieve(q)


new q I booked an appointment; what was that?
BM
BM
I booked a dentist appointment on the 15th of next month.


In [ ]:
q=prompt_text.